# Measure selection and read latency

Compare Paper greedy with tiles at coarse widths and successive halvings down to one-row Top-R.
Each method uses the same captured importance, requested row budget and direct-I/O reader.

Inputs are the profile from 01, traces from 02 and settings from
[03_01](03_01_tune_paper.ipynb). Run from this directory on the target SSD with Linux,
CUDA, PyTorch, a C++ compiler and the upstream dependencies.

The measurement begins with prepared importance on the CPU and ends with selected bytes
on the CPU. It uses a random file sized for projection weights. GPU weight transfer,
model computation and answer accuracy are outside this experiment.

In [1]:
from pathlib  import Path
from tempfile import TemporaryDirectory
from time     import perf_counter_ns

import json
import os
import platform
import subprocess
import sys
import torch

import numpy  as np
import pandas as pd

from torch.utils.cpp_extension import load

In [2]:
UPSTREAM = Path("../../vlm-flash")

sys.path.insert(0, str((UPSTREAM / "src").resolve()))
sys.path.insert(0, str((UPSTREAM / "scripts").resolve()))

import vlmflash
from vlmflash._native import native
from _lock            import exclusive

## 1. Inputs and settings

Take up to 32 evenly spaced traces per shape, preserving archive order within each shape.
For fraction $f$, select at most $R=\max(1,\lfloor fN\rfloor)$ rows.
Tiles fill the budget exactly; Paper may underfill it. At $R=N$, all methods bypass selection.

In [3]:
PROFILE = Path("../01/chunk_latency_measurements.csv")
TRACE   = Path("../02/activation_traces.json")
TUNING  = Path("paper_tuning.csv")

MAX_TRACES_PER_SHAPE = 32  # 0 uses all archived traces.
FRACTIONS            = np.arange(3, 11) / 10

WARMUP  = 3
REPEATS = 30

THREADS      = 6
MAX_READ_KIB = 768

COARSE_SCALES = [1, 1.25, 1.5, 1.75, 2]
PAPER         = "Paper greedy (GPU sort)"

In [4]:
archive = json.loads(TRACE.read_text())

weight_bytes = archive["weight_bytes"]
shapes       = sorted({(trace["n"], trace["d"]) for trace in archive["traces"]})

traces = []

for shape in shapes:
    group = []

    for trace in archive["traces"]:
        if (trace["n"], trace["d"]) == shape:
            group.append(trace)

    count   = min(len(group), MAX_TRACES_PER_SHAPE or len(group))
    indices = np.linspace(0, len(group) - 1, count, dtype=int)

    for index in indices:
        traces.append(group[index])

In [5]:
tuning = pd.read_csv(TUNING)
tuning = tuning.set_index(["n", "d"], verify_integrity=True)

assert set(shapes) <= set(tuning.index), "Run 03_01 for the missing projection shapes."

In [6]:
profile = pd.read_csv(PROFILE)
curve   = profile.groupby("size_kib").latency_ms.median()
curve   = curve.sort_index()

assert np.isfinite(curve).all() and (curve > 0).all()

boundary = float((curve / curve.index).idxmin())
ceiling  = float(curve.index.max())
table    = vlmflash.LatencyTable(curve.to_dict())

## 2. Native implementations

Paper ranks multiscale windows by importance per estimated cost on the GPU, then greedily
accepts nonoverlapping windows on the CPU. Its GPU sorting and score transfers are timed.
Tiles rank full tiles on the CPU, fill a residual interval and compare offsets zero and half a tile.
[selectors.cpp](selectors.cpp) contains the original tile implementation unchanged.
Build both implementations before measuring.

In [7]:
os.environ.setdefault("MAX_JOBS", "1")
torch.set_num_threads(1)

assert torch.cuda.is_available(), "Paper's measured GPU sort requires CUDA."

reader = native()
assert reader is not None, "Build the upstream native extension first."

selectors = load(
    "vlmflash_selection03_notebook",
    [str(Path("selectors.cpp").resolve())],
    extra_cflags = ["-O3", "-std=c++17"],
    verbose      = True,
)

[1/2] c++ -MMD -MF selectors.o.d -DTORCH_EXTENSION_NAME=vlmflash_selection03_notebook -DTORCH_API_INCLUDE_EXTENSION_H -isystem /home/yabsed/miniconda3/lib/python3.13/site-packages/torch/include -isystem /home/yabsed/miniconda3/lib/python3.13/site-packages/torch/include/torch/csrc/api/include -isystem /home/yabsed/miniconda3/include/python3.13 -fPIC -std=c++17 -O3 -std=c++17 -c /home/yabsed/Documents/fall26/work/mathmatics/graduate-paper/vlm-in-flash/final/results/03/selectors.cpp -o selectors.o 
[2/2] c++ selectors.o -shared -L/home/yabsed/miniconda3/lib/python3.13/site-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o vlmflash_selection03_notebook.so


## 3. Candidate geometry

Let $s$ minimize measured cost per KiB. Convert it to rows before taking coarse multiples
$1,1.25,1.5,1.75,2$ and successive halvings down to one row. Keep each integer width once.
This profile boundary differs from the fitted two-line boundary used in 02.

The cost lookup interpolates the measured profile and sums costs over maximal selected runs.
It chooses masks; direct-I/O times are measured separately with six reader threads.

In [8]:
def tile_grid(s):
    # Coarse tiles
    scales = []

    for scale in COARSE_SCALES:
        scales.append((f"{scale:g}s", float(scale)))

    # Finer tiles
    levels = max(0, int(np.ceil(np.log2(max(1, s)))))

    for j in range(1, levels + 1):
        scales.append((f"s/2^{j}", 2.0 ** -j))

    # Distinct widths; keep the first label.
    seen = set()
    grid = {}

    for label, scale in scales:
        width = max(1, int(np.ceil(scale * s)))
        if width in seen:
            continue

        name = "Top-R" if width == 1 else f"Tiles ({label})"
        grid[name] = (scale, width)
        seen.add(width)

    return grid

In [9]:
def paper_windows(n, w, tuned):
    if ceiling < 2 * w:
        return torch.tensor([1]), 1

    start  = max(1, int(tuned.start_kb / w))
    stride = max(1, int(tuned.jump_cap_kb / w))
    stop   = min(n + 1, int((ceiling + 1) / w))

    rows = list(range(start, stop, start))
    assert rows, "Paper window range is empty."

    return torch.tensor(rows), stride

In [10]:
def prepare(trace):
    # Importance
    values = torch.tensor(trace["values"], dtype=torch.float32)
    assert values.ndim == 1 and torch.isfinite(values).all() and (values >= 0).all()

    # Read cost
    n       = trace["n"]
    row_kib = trace["row_bytes"] / 1024
    costs   = [0.0]

    for r in range(1, n + 1):
        costs.append(table.read_ms(row_kib * r))

    costs = torch.tensor(costs, dtype=torch.float64)

    # Paper
    d     = trace["d"]
    tuned = tuning.loc[(n, d)]

    sizes, stride = paper_windows(n, row_kib, tuned)
    delays        = costs[sizes]

    methods = {
        PAPER: lambda R: reader.select_chunks(
            values, R, sizes, delays, stride, True,
        )[0],
    }

    # Tiles
    grid = tile_grid(boundary / row_kib)

    for name, (scale, width) in grid.items():
        methods[name] = lambda R, B=width: selectors.tiles(values, R, costs, B)[0]

    return values, costs, methods, grid

## 4. Case information

A case is one trace, requested budget and method. Save its mask and metadata once.
Retained importance is $100\sum_i v_iM_i/\sum_i v_i$; fill is $100|M|/R$.
Importance is the captured mean absolute activation, a proxy for answer quality.
Intervals are maximal contiguous runs stored as `[start, end)` pairs.

In [11]:
def chunks(mask):
    """Return selected chunks as [start, end) intervals."""

    mask   = mask.numpy().astype(bool)
    padded = np.concatenate(([0], mask, [0]))
    change = np.diff(padded)

    starts = np.flatnonzero(change ==  1) # 0 -> 1
    ends   = np.flatnonzero(change == -1) # 1 -> 0

    return np.column_stack((starts, ends))

In [12]:
def describe(mask, values, R, costs, method):
    # Selected rows
    selected = int(mask.sum())

    if method == PAPER:
        assert selected <= R
    else:
        assert selected == R

    # Retained importance
    total    = float(values.double().sum())
    retained = np.nan

    if total:
        importance = float(values[mask.bool()].double().sum())
        retained   = 100 * importance / total

    # Read cost
    spans   = chunks(mask)
    lengths = np.diff(spans).ravel()
    latency = float(costs[lengths].sum())

    return {
        "R_selected"    : selected,
        "fill_pct"      : 100 * selected / R,
        "retained_pct"  : retained,
        "modeled_io_ms" : latency,
        "runs"          : len(spans),
        "intervals"     : json.dumps(spans.tolist()),
    }

## 5. Timing

Measure selection and selection-through-read wall time with the same outer clock.
Native `io_ms` covers its read/thread interval; wall time also includes read preparation.
Adjacent rows merge, and requests split at 768 KiB. Require `O_DIRECT` so the measurements
bypass the operating system's page cache. Validation occurs after the timed interval.

In [13]:
def timed_read(select, R, expected, blob, row_bytes):
    full = R == expected.numel()

    # Selection
    begin        = perf_counter_ns()
    mask         = expected if full else select(R)
    after_select = perf_counter_ns()

    # Read
    raw, io_us, _, direct = reader.read_rows(
        blob, mask, row_bytes, "cpu", THREADS, MAX_READ_KIB * 1024, True,
    )
    end = perf_counter_ns()

    # Outside timing
    assert direct, "O_DIRECT unavailable; refusing page-cache timings."
    assert torch.equal(mask, expected)
    assert raw.shape == (int(expected.sum()), row_bytes)

    return {
        "selection_ms" : (after_select - begin) / 1e6,
        "io_ms"        : io_us / 1000,
        "wall_ms"      : (end - begin) / 1e6,
    }

In [14]:
def measure_trace(trace, blob, rng):
    values, costs, methods, grid = prepare(trace)

    n     = len(values)
    dense = torch.ones(n, dtype=torch.uint8)

    for fraction in FRACTIONS:
        R       = max(1, int(round(float(fraction), 6) * n))
        checked = {}

        # Selections
        for name, select in methods.items():
            mask         = dense if R == n else select(R)
            scale, width = grid.get(name, (np.nan, np.nan))

            if width == 1:
                order = torch.argsort(values, descending=True, stable=True)
                top   = torch.zeros(n, dtype=torch.uint8)
                top[order[:R]] = 1

                assert torch.equal(mask, top), "One-row tiles must equal stable Top-R."

            case_id = len(case_records)

            case_records.append({
                # Trace
                "case_id"  : case_id,
                "trace_id" : trace["trace_id"],
                "n"        : n,
                "d"        : trace["d"],

                # Requested budget
                "fraction"  : float(fraction),
                "R_nominal" : R,

                # Selection method
                "tile_scale" : scale,
                "tile_rows"  : width,
                "tile_kib"   : width * trace["row_bytes"] / 1024,
                "method"     : name,

                # Selected mask
                **describe(mask, values, R, costs, name),
            })

            checked[name] = (case_id, mask.clone())

        # Repeated measurements
        for repeat in range(-WARMUP, REPEATS):
            for name in rng.permutation(list(methods)):
                case_id, mask = checked[name]
                timing = timed_read(
                    methods[name], R, mask, blob, trace["row_bytes"],
                )

                if repeat >= 0:  # Exclude warmup rounds.
                    timing_records.append({
                        "case_id" : case_id,
                        "repeat"  : repeat,
                        **timing,
                    })

## 6. Measurement

Allocate and flush a random file on the target filesystem before timing.
Randomize method order within each trace and budget, including warmup rounds.
The lock prevents cooperating experiments from measuring this device concurrently.

In [15]:
def write_blob(path, size):
    with open(path, "wb") as stream:
        os.posix_fallocate(stream.fileno(), 0, size)

        for begin in range(0, size, 2**20):
            count = min(2**20, size - begin)
            stream.write(os.urandom(count))

        stream.flush()
        os.fsync(stream.fileno())

In [16]:
rng = np.random.default_rng(0)

case_records   = []
timing_records = []

with exclusive("experiment03"), TemporaryDirectory(dir=Path.cwd()) as work:
    blob = str(Path(work) / "rows.dat")
    size = max(trace["n"] * trace["row_bytes"] for trace in traces)
    write_blob(blob, size)

    for index, trace in enumerate(traces, 1):
        measure_trace(trace, blob, rng)

        if index % 16 == 0 or index == len(traces):
            print(f"{index}/{len(traces)} traces measured", flush=True)

16/128 traces measured
32/128 traces measured
48/128 traces measured
64/128 traces measured
80/128 traces measured
96/128 traces measured
112/128 traces measured
128/128 traces measured


## 7. Output

`cases.csv` contains one row per case; `timings.csv` contains its repeat samples.
`case_id` joins the two files. Store acquisition settings and the selected tuning snapshot
in `environment.json`, so later tuning changes do not change interpretation of this run.
The save cells replace these three files only after the measurement completes.

In [17]:
cases   = pd.DataFrame(case_records)
timings = pd.DataFrame(timing_records)

assert cases.case_id.is_unique
assert not timings.duplicated(["case_id", "repeat"]).any()
assert timings.groupby("case_id").size().eq(REPEATS).all()

In [18]:
paper_tuning = []

for n, d in shapes:
    setting = tuning.loc[(n, d)]

    best = {
        "start_kb"    : int(setting.start_kb),
        "jump_cap_kb" : int(setting.jump_cap_kb),
        "overhead_ms" : float(setting.overhead_ms),
    }

    paper_tuning.append({
        "n"             : n,
        "d"             : d,
        "best"          : best,
        "num_feasible"  : int(setting.num_feasible),
        "num_evaluated" : int(setting.num_evaluated),
    })

In [19]:
upstream_commit = subprocess.check_output(
    ["git", "-C", str(UPSTREAM), "rev-parse", "HEAD"],
    text = True,
).strip()

mount = subprocess.check_output(
    ["findmnt", "-T", str(Path.cwd()), "-n", "-o", "SOURCE,FSTYPE,TARGET"],
    text = True,
).strip()

In [20]:
environment = {
    # Device and implementation
    "platform"        : platform.platform(),
    "torch"           : torch.__version__,
    "cuda_available"  : torch.cuda.is_available(),
    "upstream_commit" : upstream_commit,
    "mount"           : mount,

    # Selection and reader
    "paper_sort_devices" : ["cuda"],
    "paper_tuning"       : paper_tuning,
    "torch_threads"      : torch.get_num_threads(),
    "reader_threads"     : THREADS,
    "direct_io"          : True,
    "destination"        : "cpu",
    "max_read_kib"       : MAX_READ_KIB,

    # Candidate geometry
    "tile_boundary_kib" : boundary,
    "coarse_scales"     : COARSE_SCALES,
    "dtype_bytes"       : weight_bytes,
    "tile_grid"         : "halving_to_one_row; unique widths per shape",
    "tile_offsets"      : "0, floor(B/2)",
    "paper_ceiling_kib" : ceiling,

    # Measurement
    "traces"      : len(traces),
    "warmup"      : WARMUP,
    "repeats"     : REPEATS,
    "fractions"   : FRACTIONS.tolist(),
    "measurement" : "prepared CPU importance through selected bytes on CPU; no model compute",

    # Output
    "cases_file"   : "cases.csv",
    "timings_file" : "timings.csv",
    "cases"        : len(cases),
    "samples"      : len(timings),
}

In [21]:
cases.to_csv("cases.csv", index=False)
timings.to_csv("timings.csv", index=False)

Path("environment.json").write_text(json.dumps(environment, indent=2) + "\n")
print(f"Saved {len(cases):,} cases and {len(timings):,} repeat samples.")

Saved 14,080 cases and 422,400 repeat samples.
